In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
import os
import sys
import subprocess

In [ ]:
def set_env(input_dir):

    wheels_dir = f"{input_dir}/wheels"

    if not os.path.isdir(wheels_dir):
        raise FileNotFoundError(f"Wheels folder not found: {wheels_dir}")

    if not os.listdir(wheels_dir):
        raise RuntimeError(f"Wheels folder is empty: {wheels_dir}")

    subprocess.run([
        sys.executable,
        '-m',
        'pip',
        'install',
        '--no-index',
        '--find-links',
        wheels_dir,
        'unsloth',
        'trl',
        'vllm',
        'openai_harmony'
    ], check=True)

In [ ]:
set_env(
    input_dir='/kaggle/input/datasets/ramkumar86/aimo-wheels-cache'
)

In [ ]:
subprocess.run(['ls', '/kaggle/input/datasets/ramkumar86/aimo-wheels-cache/tiktoken_encodings'])

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/input/datasets/ramkumar86/aimo-wheels-cache/tiktoken_encodings'

In [ ]:
import gc
import re
import math
import json
import time
import queue
import threading
import contextlib
from pathlib import Path
from datetime import datetime, timezone
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

try:
    import psutil
except ImportError:
    psutil = None

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [ ]:
class CFG:
    
    system_prompt = (
        'You are an expert mathematical problem solver at IMO level. '
        'Solve the given problem with rigorous reasoning and verify your answer.\n\n'

        '# Answer Format\n'
        'The answer is always a non-negative integer in [0, 99999].\n'
        'Do NOT apply modulo or any reduction to your final answer — '
        'the exact answer already fits in this range.\n'
        'Write your final answer as \\boxed{N}.\n\n'

        '# Python Sandbox\n'
        'Pre-loaded: sympy, numpy, math, itertools, collections, mpmath '
        '(precision=64 digits).\n'
        'Always use print() to see output. Variables persist between calls.\n\n'

        '# Critical Rules\n'
        '- Verify your answer on small cases before finalising\n'
        '- If your result is not an integer in [0, 99999], you made an error\n'
        '- Use code to verify, not to replace reasoning'
    )

    tool_prompt = (
        'Execute Python code for calculations, verification, and enumeration.\n'
        'Pre-loaded: sympy, numpy, math, itertools, collections, mpmath.\n'
        'Always print() results. Fix errors and retry with a different approach.'
    )
    
    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1'
    speculative_model_path = '/kaggle/input/datasets/ramkumar86/gpt-oss-120b-p-eagle'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    stream_interval = 20
    context_tokens = 24576
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 32
    max_num_batched_tokens = 8192
    max_cudagraph_capture_size = 2048
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.90
    speculative_tokens = 3
    temperature = 1.0
    min_p = 0.02
    log_path = 'aimo3_speculative_problem_logs.jsonl'
    summary_path = 'aimo3_speculative_problem_summary.csv'

    max_tool_output_chars = 5000
    max_tool_output_head_chars = 3200
    max_tool_output_tail_chars = 1400
    error_tool_output_head_chars = 1500
    error_tool_output_tail_chars = 3000


In [ ]:
set_seed(CFG.seed)

In [ ]:
class AIMO3Template:

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [ ]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

In [ ]:
class AIMO3Tool:

    def __init__(
        self,
        local_jupyter_timeout: float,
        tool_prompt: str,
        sandbox=None,
        max_output_chars: int | None = None,
        max_output_head_chars: int | None = None,
        max_output_tail_chars: int | None = None,
        error_output_head_chars: int | None = None,
        error_output_tail_chars: int | None = None
    ):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        self._max_output_chars = max_output_chars
        self._max_output_head_chars = max_output_head_chars
        self._max_output_tail_chars = max_output_tail_chars
        self._error_output_head_chars = error_output_head_chars
        self._error_output_tail_chars = error_output_tail_chars
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _truncate_output(self, output: str) -> str:

        if self._max_output_chars is None or len(output) <= self._max_output_chars:
            return output

        is_error_output = (
            output.startswith('[ERROR]')
            or 'Traceback' in output
            or 'Error:' in output
        )

        if is_error_output:
            head_chars = self._error_output_head_chars
            tail_chars = self._error_output_tail_chars
        else:
            head_chars = self._max_output_head_chars
            tail_chars = self._max_output_tail_chars

        marker = f'\n...[truncated, {len(output)} chars total]\n'

        if head_chars is None or tail_chars is None:
            return output[:self._max_output_chars] + marker.rstrip()

        head_chars = max(0, head_chars)
        tail_chars = max(0, tail_chars)

        if head_chars + tail_chars >= len(output):
            return output

        available = max(0, self._max_output_chars - len(marker))

        if available == 0:
            return marker.rstrip()

        head_chars = min(head_chars, available, len(output))
        remaining = max(0, available - head_chars)
        tail_chars = min(tail_chars, remaining, max(0, len(output) - head_chars))

        if head_chars + tail_chars >= len(output):
            return output

        head = output[:head_chars]
        tail = output[-tail_chars:] if tail_chars > 0 else ''

        return head + marker + tail

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=self._truncate_output(output))
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]


In [ ]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):
    
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
    
        self._preload_model_weights()
        
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
    
        self._wait_for_server()
        self._initialize_kernels()
    
        self.notebook_start_time = time.time()
        self.problems_remaining = 50
        self.problem_counter = 0
        self.run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        self.log_path = Path(self.cfg.log_path)
        self.summary_path = Path(self.cfg.summary_path)
        self.kv_cache_stats = self._get_kv_cache_stats()
        self.log_path.write_text('', encoding='utf-8')

        if self.summary_path.exists():
            self.summary_path.unlink()
    

    def _get_memory_usage_mb(self) -> dict:

        stats = {'solver_rss_mb': None, 'server_rss_mb': None}

        if psutil is None:
            return stats

        try:
            stats['solver_rss_mb'] = round(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2), 2)
        except Exception:
            pass

        try:
            if self.server_process.poll() is None:
                stats['server_rss_mb'] = round(psutil.Process(self.server_process.pid).memory_info().rss / (1024 ** 2), 2)
        except Exception:
            pass

        return stats

    def _get_gpu_memory_stats_mb(self) -> dict:

        stats = {
            'gpu_memory_used_mb': None,
            'gpu_memory_free_mb': None,
            'gpu_memory_total_mb': None,
            'gpu_utilization_pct': None,
            'gpu_memory_utilization_pct': None
        }

        try:
            output = subprocess.check_output(
                [
                    'nvidia-smi',
                    '--query-gpu=memory.used,memory.free,memory.total,utilization.gpu,utilization.memory',
                    '--format=csv,noheader,nounits'
                ],
                text=True,
                stderr=subprocess.DEVNULL
            ).strip()
        except Exception:
            return stats

        if not output:
            return stats

        parts = [part.strip() for part in output.splitlines()[0].split(',')]

        if len(parts) != 5:
            return stats

        for key, value in zip(list(stats.keys()), parts):
            try:
                stats[key] = round(float(value), 2)
            except Exception:
                stats[key] = None

        return stats

    def _get_kv_cache_stats(self) -> dict:

        stats = {
            'kv_cache_memory_gib': None,
            'kv_cache_tokens': None,
            'kv_cache_max_concurrency': None
        }

        log_path = Path('vllm_server.log')

        if not log_path.exists():
            return stats

        try:
            log_text = log_path.read_text(encoding='utf-8', errors='ignore')
        except Exception:
            return stats

        patterns = {
            'kv_cache_memory_gib': r'Available KV cache memory:\s*([0-9.]+)\s*GiB',
            'kv_cache_tokens': r'GPU KV cache size:\s*([0-9,]+)\s*tokens',
            'kv_cache_max_concurrency': r'Maximum concurrency for\s*[0-9,]+\s*tokens per request:\s*([0-9.]+)x'
        }

        for key, pattern in patterns.items():
            matches = re.findall(pattern, log_text)
            if not matches:
                continue
            value = matches[-1].replace(',', '')
            try:
                stats[key] = int(value) if key == 'kv_cache_tokens' else round(float(value), 4)
            except Exception:
                pass

        return stats

    def _log_problem_stats(self, payload: dict) -> None:

        with self.log_path.open('a', encoding='utf-8') as file_object:
            file_object.write(json.dumps(payload, ensure_ascii=True) + '\n')

        summary_row = {
            key: payload.get(key)
            for key in [
                'run_id', 'problem_number', 'problem_id', 'final_answer', 'elapsed_seconds',
                'budget_seconds', 'attempts_completed', 'attempts_with_answers',
                'total_prompt_tokens', 'total_completion_tokens', 'total_python_calls',
                'total_python_errors', 'solver_rss_mb', 'server_rss_mb',
                'gpu_memory_used_mb', 'gpu_memory_free_mb', 'gpu_memory_total_mb',
                'gpu_utilization_pct', 'gpu_memory_utilization_pct',
                'kv_cache_memory_gib', 'kv_cache_tokens', 'kv_cache_max_concurrency'
            ]
        }

        pd.DataFrame([summary_row]).to_csv(
            self.summary_path,
            mode='a',
            header=not self.summary_path.exists(),
            index=False
        )

    def _preload_model_weights(self) -> None:
    
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0
    
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
    
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
    
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
    
    def _start_server(self) -> subprocess.Popen:
    
        os.environ['VLLM_USE_FLASHINFER_MOE_MXFP4_MXFP8'] = '1'
        os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    
        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--max-num-batched-tokens', 
            str(self.cfg.max_num_batched_tokens), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--max-cudagraph-capture-size', 
            str(self.cfg.max_cudagraph_capture_size), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--no-enable-prefix-caching', 
            '--speculative-config', 
            f'{{"method":"eagle3","model":"{self.cfg.speculative_model_path}","num_speculative_tokens":{self.cfg.speculative_tokens},"parallel_drafting":true}}'
        ]
    
        self.log_file = open('vllm_server.log', 'w')
    
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )
    
    def _wait_for_server(self):
    
        print('Waiting for vLLM server...')
        start_time = time.time()
    
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
    
            if return_code is not None:
                self.log_file.flush()
    
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
    
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
    
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
    
                return
    
            except Exception:
                time.sleep(1)
    
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self) -> None:
    
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
    
        self.sandbox_pool = queue.Queue()
    
        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
    
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
    
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
    
    def _scan_for_answer(self, text: str) -> int | None:
        
        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
                
        pattern = r'final\s+answer\s+is\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
    
        return None
    
    def _compute_mean_entropy(self, logprobs_buffer: list) -> float:
    
        if not logprobs_buffer:
            return float('inf')
    
        total_entropy = 0.0
        token_count = 0
    
        for top_logprobs_dict in logprobs_buffer:
            
            if not isinstance(top_logprobs_dict, dict):
                continue
            
            if not top_logprobs_dict:
                continue
            
            token_entropy = 0.0
            
            for token_str, log_prob in top_logprobs_dict.items():
                prob = math.exp(log_prob)
                
                if prob > 0:
                    token_entropy -= prob * math.log2(prob)
            
            total_entropy += token_entropy
            token_count += 1
    
        if token_count == 0:
            return float('inf')
    
        return total_entropy / token_count
    
    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float
    ) -> dict:
    
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1,
                'Prompt Tokens': 0,
                'Completion Tokens': 0,
                'Response Length': 0,
                'Python Calls': 0,
                'Python Errors': 0,
                'Entropy': float('inf'),
                'Elapsed Seconds': 0.0,
                'Tokens / Second': 0.0,
                'Answer': None
            }
    
        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        prompt_tokens = 0
        final_answer = None
        attempt_start_time = time.time()
        
        logprobs_buffer = []
    
        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))
    
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
    
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox,
                max_output_chars=self.cfg.max_tool_output_chars,
                max_output_head_chars=self.cfg.max_tool_output_head_chars,
                max_output_tail_chars=self.cfg.max_tool_output_tail_chars,
                error_output_head_chars=self.cfg.error_tool_output_head_chars,
                error_output_tail_chars=self.cfg.error_tool_output_tail_chars
            )
    
            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, 
                problem, 
                local_tool.tool_config
            )
    
            conversation = Conversation.from_messages(messages)
    
            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break
    
                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                prompt_tokens += len(prompt_ids)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
    
                if max_tokens < self.cfg.buffer_tokens:
                    break
    
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name, 
                    temperature=self.cfg.temperature, 
                    logprobs=self.cfg.top_logprobs, 
                    max_tokens=max_tokens, 
                    prompt=prompt_ids, 
                    seed=attempt_seed, 
                    stream=True, 
                    extra_body={
                        'stop_token_ids': self.stop_token_ids, 
                        'return_token_ids': True
                    }
                )
    
                try:
                    token_buffer = []
                    text_chunks = []
    
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break
    
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text
    
                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)
                            
                            chunk_logprobs = chunk.choices[0].logprobs
                            
                            if chunk_logprobs is not None:
                                if chunk_logprobs.top_logprobs:
                                    logprobs_buffer.extend(chunk_logprobs.top_logprobs)
    
                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)
    
                            if answer is not None:
                                final_answer = answer
                                break
    
                finally:
                    stream.close()
    
                if final_answer is not None:
                    break
    
                if not token_buffer:
                    break
    
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]
    
                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = self._scan_for_answer(answer_text)
                    break
    
                if last_message.recipient == 'python':
                    python_calls += 1
                    tool_responses = local_tool.process_sync_plus(last_message)
    
                    response_text = tool_responses[0].content[0].text
    
                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
                        python_errors += 1
    
                    conversation.messages.extend(tool_responses)
    
        except Exception as exc:
            python_errors += 1
    
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
    
        mean_entropy = self._compute_mean_entropy(logprobs_buffer)
        elapsed_seconds = time.time() - attempt_start_time
        tokens_per_second = total_tokens / elapsed_seconds if elapsed_seconds > 0 else 0.0
    
        return {
            'Attempt': attempt_index + 1,
            'Prompt Tokens': prompt_tokens,
            'Completion Tokens': total_tokens,
            'Response Length': total_tokens,
            'Python Calls': python_calls,
            'Python Errors': python_errors,
            'Entropy': mean_entropy,
            'Elapsed Seconds': round(elapsed_seconds, 3),
            'Tokens / Second': round(tokens_per_second, 3),
            'Answer': final_answer
        }
    
    def _select_answer(self, detailed_results: list) -> int:

        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']
            
            if answer is not None:
                weight = 1.0 / max(entropy, 1e-9)
                
                answer_weights[answer] += weight
                answer_votes[answer] += 1

        scored_answers = []

        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer, 
                'votes': answer_votes[answer], 
                'score': total_weight
            })

        scored_answers.sort(key=lambda x: x['score'], reverse=True)

        vote_data = []

        for item in scored_answers:
            vote_data.append((
                item['answer'], 
                item['votes'], 
                item['score']
            ))

        vote_dataframe = pd.DataFrame(
            vote_data, 
            columns=['Answer', 'Votes', 'Score']
        )

        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)
        
        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0

        final_answer = scored_answers[0]['answer']    
        print(f'\nFinal Answer: {final_answer}\n')

        return final_answer
    
    def solve_problem(self, problem: str, problem_id: str | None = None) -> int:
    
        self.problem_counter += 1
        problem_label = problem_id if problem_id is not None else f'problem_{self.problem_counter}'

        print(f'\nProblem #{self.problem_counter} (ID: {problem_label}): {problem}\n')
        
        user_input = f'{problem}'
    
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
    
        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
    
        deadline = time.time() + budget
        problem_start_time = time.time()
    
        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')
        print(f'KV Cache Stats: {self.kv_cache_stats}')
    
        tasks = []
    
        for attempt_index in range(self.cfg.attempts):
            tasks.append((self.cfg.system_prompt, attempt_index))
    
        detailed_results = []
        valid_answers = []
    
        stop_event = threading.Event()
    
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
    
        try:
            futures = []
    
            for (system_prompt, attempt_index) in tasks:
                future = executor.submit(
                    self._process_attempt, 
                    user_input, 
                    system_prompt, 
                    attempt_index, 
                    stop_event, 
                    deadline
                )
    
                futures.append(future)
    
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
    
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
    
                    counts = Counter(valid_answers).most_common(1)
    
                    if counts and counts[0][1] >= self.cfg.early_stop:
                        stop_event.set()
    
                        for f in futures:
                            f.cancel()
    
                        break
    
                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue
    
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
            
            self.problems_remaining = max(0, self.problems_remaining - 1)
    
        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Entropy'] = results_dataframe['Entropy'].round(3)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')
            
            display(results_dataframe)
    
        elapsed_seconds = round(time.time() - problem_start_time, 3)

        if not valid_answers:
            final_answer = 0
            print('\nResult: 0\n')
        else:
            final_answer = self._select_answer(detailed_results)

        memory_stats = self._get_memory_usage_mb()
        gpu_stats = self._get_gpu_memory_stats_mb()

        payload = {
            'run_id': self.run_id,
            'problem_number': self.problem_counter,
            'problem_id': problem_label,
            'final_answer': final_answer,
            'elapsed_seconds': elapsed_seconds,
            'budget_seconds': round(budget, 3),
            'attempts_completed': len(detailed_results),
            'attempts_with_answers': sum(1 for result in detailed_results if result['Answer'] is not None),
            'total_prompt_tokens': int(sum(result.get('Prompt Tokens', 0) for result in detailed_results)),
            'total_completion_tokens': int(sum(result.get('Completion Tokens', 0) for result in detailed_results)),
            'total_python_calls': int(sum(result.get('Python Calls', 0) for result in detailed_results)),
            'total_python_errors': int(sum(result.get('Python Errors', 0) for result in detailed_results)),
            **memory_stats,
            **gpu_stats,
            **self.kv_cache_stats,
            'attempt_details': detailed_results
        }

        self._log_problem_stats(payload)

        print(f'Time Taken: {elapsed_seconds:.2f} seconds')
        print(f'Logs written to: {self.log_path} and {self.summary_path}\n')

        return final_answer
    
    def __del__(self):
    
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
    
        if hasattr(self, 'log_file'):
            self.log_file.close()
    
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
    
                except Exception:
                    pass

In [ ]:
solver = AIMO3Solver(CFG)

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    
    id_value = id_.item(0)
    question_text = question.item(0)
    
    gc.disable()
    
    final_answer = solver.solve_problem(question_text, problem_id=str(id_value))
    
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )